# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset


In [13]:
import pandas as pd
import os

# Define the file path. Using the backup from GitHub if the local cleaned file is missing.
file_path = 'cleaned_airbnb_data_7.csv'
backup_url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/airbnb_listings.csv'

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Successfully loaded local file: {file_path}")
else:
    print(f"{file_path} not found. Loading backup dataset from GitHub...")
    df = pd.read_csv(backup_url)
    print("Successfully loaded backup dataset.")

# Display basic information and a preview
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.head()

cleaned_airbnb_data_7.csv not found. Loading backup dataset from GitHub...
Successfully loaded backup dataset.
Dataset Shape: 459 rows, 77 columns



,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,host_url,host_name,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20250804133828,2025-08-04,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://www.airbnb.com/users/show/4621559,Kenneth,...,4.56,3.22,3.67,NaN,0,1,1,0,0,0.07
1,3820211,https://www.airbnb.com/rooms/3820211,20250804133828,2025-08-04,city scrape,Funky Urban Gem: Prime Central Location - Park...,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.81,4.81,4.77,NaN,0,4,4,0,0,2.32
2,5651579,https://www.airbnb.com/rooms/5651579,20250804133828,2025-08-04,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",The neighborhood is very eclectic. We have a v...,https://www.airbnb.com/users/show/29288920,Gregg,...,4.88,4.76,4.64,NaN,0,2,1,1,0,2.97
3,6623339,https://www.airbnb.com/rooms/6623339,20250804133828,2025-08-04,city scrape,Bright & Cozy City Stay · Top Location + Parking!,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.70,4.80,4.72,NaN,0,4,4,0,0,2.68
4,9005989,https://www.airbnb.com/rooms/9005989,20250804133828,2025-08-04,city scrape,"Studio in The heart of Center SQ, in Albany NY",(21 years of age or older ONLY) NON- SMOKING.....,"There are many shops, restaurants, bars, museu...",https://www.airbnb.com/users/show/17766924,Sugey,...,4.93,4.87,4.77,NaN,0,1,1,0,0,5.67


### Reflection:
1. What business information does the dataset include?
2. How many rows and columns are present?

### ✍️ Your Response: 🔧
1. It includes details about Airbnb hosts, listings, locations, property features, booking options, and review scores.

2. It has 459 rows and 77 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

In [14]:
# Identify columns that might correspond to IDs, URLs, text fields, or details in our loaded dataframe
columns_to_drop = [
    'id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
    'description', 'neighborhood_overview', 'host_id', 'host_url',
    'host_name', 'host_since', 'host_location', 'host_about',
    'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood',
    'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
    'city', 'state', 'zipcode', 'market', 'smart_location', 'country_code',
    'country', 'property_type', 'room_type', 'bed_type', 'amenities',
    'weekly_price', 'monthly_price', 'calendar_updated', 'has_availability',
    'calendar_last_scraped', 'first_review', 'last_review', 'license',
    'instant_bookable', 'is_business_travel_ready', 'cancellation_policy',
    'require_guest_profile_picture', 'require_guest_phone_verification'
]

# Filter list to only columns that actually exist in df
existing_cols_to_drop = [col for col in columns_to_drop if col in df.columns]

# Drop selected columns
df_cleaned = df.drop(columns=existing_cols_to_drop)

# Save the cleaned dataframe back into df
df = df_cleaned

# Verify the remaining columns
print(f"New Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("Remaining columns:")
print(df.columns.tolist()[:20])

New Dataset Shape: 459 rows, 48 columns

Remaining columns:
['host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights']


### Reflection:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?

### ✍️ Your Response: 🔧
1. I dropped URLs, unique identifiers, text descriptions, and scraping metadata because they are non-numeric and cannot be directly parsed by a regression model without advanced transformation.

2. Including them could cause modeling errors, overfit the model to unique IDs, or add noise that reduces the accuracy of our price predictions.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

In [15]:
# Select only numeric columns for correlation analysis
numeric_df = df.select_dtypes(include=['number'])

# Compute correlation matrix
corr_matrix = numeric_df.corr()

# Get correlation of all numeric features with price, sorted in descending order
price_correlations = corr_matrix['price'].sort_values(ascending=False)

print("Correlation of numeric variables with Price:")
print(price_correlations)

Correlation of numeric variables with Price:
price                                           1.000000
accommodates                                    0.579588
beds                                            0.547032
bedrooms                                        0.499286
bathrooms                                       0.468030
estimated_revenue_l365d                         0.249488
maximum_maximum_nights                          0.122872
minimum_maximum_nights                          0.112166
maximum_nights_avg_ntm                          0.111271
availability_30                                 0.108409
availability_60                                 0.060509
availability_90                                 0.040997
calculated_host_listings_count_entire_homes     0.033206
availability_eoy                                0.032537
review_scores_value                             0.018269
maximum_nights                                  0.017992
calculated_host_listings_count             

### Reflection:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?

### ✍️ Your Response: 🔧
1. Accommodates, beds, bedrooms, and bathrooms had the strongest positive correlations with price. Review scores for communication and the longitude coordinate had the strongest negative correlations.

2. Features measuring property capacity, such as accommodates, beds, bedrooms, and bathrooms, seem like the most useful predictors.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

In [16]:
# Drop non-numeric columns and missing values first to prepare clean features for the model
numeric_df = df.select_dtypes(include=['number']).dropna()

# Define the target variable (y)
y = numeric_df['price']

# Define the features (X) by removing the price column
X = numeric_df.drop(columns=['price'])

print(f"Target shape: {y.shape}")
print(f"Features shape: {X.shape}")

Target shape: (366,)
Features shape: (366, 39)


### Reflection:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?

### ✍️ Your Response: 🔧
1. I am using the 39 numeric columns left after dropping the non-modeling columns and rows with missing values, including features like accommodates, beds, bedrooms, bathrooms, and availability metrics.

2. It is a regression problem because we are predicting a continuous numerical value (price) rather than predicting discrete, categorical classes.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [17]:
# Split the dataset into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Testing set shape:  X_test={X_test.shape}, y_test={y_test.shape}")

Training set shape: X_train=(292, 39), y_train=(292,)
Testing set shape:  X_test=(74, 39), y_test=(74,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [18]:
# Initialize the Linear Regression model
model = LinearRegression()

# Fit the model using the training data
model.fit(X_train, y_train)

# Predict the prices for the test set
y_pred = model.predict(X_test)

print("Model fitting completed and predictions generated on the testing set.")

Model fitting completed and predictions generated on the testing set.


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

In [19]:
# Calculate evaluation metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2) Score:     {r2:.4f}")

Mean Squared Error (MSE): 14791.50
R-squared (R2) Score:     0.2347


### Reflection:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?

### ✍️ Your Response: 🔧
1. The R-squared score is approximately 0.2347, which means the model only explains about 23.5% of the variation in Airbnb listing prices. This indicates a relatively weak overall fit.

2. The MSE of 14791.50 is quite large, meaning our predictions deviate significantly on average from the actual prices. We could improve it by handling outliers, incorporating categorical variables like room type via encoding, or using non-linear models.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

In [20]:
# Add code here 🔧

In [21]:
# Create a DataFrame for feature names and their regression coefficients
coefficients_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_,
    'Absolute_Coefficient': abs(model.coef_)
})

# Sort the table by the absolute value of the coefficient to highlight the most impactful features at the top
coefficients_df = coefficients_df.sort_values(by='Absolute_Coefficient', ascending=False).drop(columns=['Absolute_Coefficient'])

# Reset the index for a cleaner table layout
coefficients_df = coefficients_df.reset_index(drop=True)

# Display the table
coefficients_df

,Feature,Coefficient
0,latitude,-298.390431
1,longitude,-197.969335
2,review_scores_value,38.860352
3,bathrooms,33.835345
4,review_scores_communication,-31.215357
5,maximum_minimum_nights,-20.494860
6,review_scores_accuracy,20.468990
7,minimum_nights_avg_ntm,18.031631
8,accommodates,16.159835
9,review_scores_checkin,-13.884298


### Reflection:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?

### ✍️ Your Response: 🔧
1. Review scores for value, the number of bathrooms, the average minimum nights, and the accommodation capacity increased the price the most among our standard features.

2. Latitude and longitude both had highly negative coefficients, which is surprising at first, but it indicates that moving in certain geographic directions has a massive impact on price in this specific market.

3. This tells us that physical size (bathrooms, capacity) and high perceived value reviews are important, but location coordinates are the strongest predictors of price variations within the region.

## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

In [22]:
# 1. Select the top 5 features with the strongest absolute coefficients
top_features = ['latitude', 'longitude', 'review_scores_value', 'bathrooms', 'review_scores_communication']

# 2. Rebuild datasets using only these top features
X_train_refined = X_train[top_features]
X_test_refined = X_test[top_features]

# 3. Train the refined model
refined_model = LinearRegression()
refined_model.fit(X_train_refined, y_train)

# 4. Predict prices using the refined model
y_pred_refined = refined_model.predict(X_test_refined)

# 5. Evaluate and compare performance
mse_refined = mean_squared_error(y_test, y_pred_refined)
r2_refined = r2_score(y_test, y_pred_refined)

print("Baseline Model (All Features):")
print(f"  Mean Squared Error (MSE): {mse:.2f}")
print(f"  R-squared (R2) Score:     {r2:.4f}\n")

print("Refined Model (Top 5 Features):")
print(f"  Mean Squared Error (MSE): {mse_refined:.2f}")
print(f"  R-squared (R2) Score:     {r2_refined:.4f}")

Baseline Model (All Features):
  Mean Squared Error (MSE): 14791.50
  R-squared (R2) Score:     0.2347

Refined Model (Top 5 Features):
  Mean Squared Error (MSE): 20339.29
  R-squared (R2) Score:     -0.0524


### Reflection:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?

### ✍️ Your Response: 🔧
1. I kept latitude, longitude, review_scores_value, bathrooms, and review_scores_communication because they had the largest absolute coefficients in the baseline regression.

2. No, the performance got significantly worse. The MSE increased to 20339.29 and the R-squared score dropped to -0.0524. This happened because removing all other features lost important supporting information and led to an underfitted model that performs worse than a simple horizontal line average on the test set.

3. I would recommend the baseline model over the refined one, though ideally we should explore non-linear algorithms or better feature engineering since neither linear regression model performs exceptionally well.

4. This directly relates to my learning outcome of understanding how feature selection affects predictive performance and learning that choosing features solely by individual coefficients does not guarantee a better multivariate model.

## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### Reflection:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. The model helped answer how physical characteristics like capacity and bathrooms, combined with geographic coordinates, impact the listing price of an Airbnb.

2. I would recommend that hosts focus on maximizing listing capacity and bathrooms to justify higher prices, while ensuring they keep value reviews high since perceived value positively relates to pricing success.

3. To improve this model, we should handle extreme outliers in prices, encode categorical columns like room type or neighborhood, and test more complex non-linear models like Random Forests or Gradient Boosting.

4. This relates to my learning outcome by demonstrating how to interpret regression metrics and translate raw coefficients into actionable real-world pricing strategies.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [24]:
# Convert the notebook to HTML format for submission
!jupyter nbconvert --to html /content/assignment_11_regression.ipynb

[NbConvertApp] Converting notebook /content/assignment_11_regression.ipynb to html
[NbConvertApp] Writing 359905 bytes to /content/assignment_11_regression.html
